# 국자경 lane/da 라벨링 pre-label 파이프라인

**흐름**
1. **da(주행가능영역)**: SAM에 클릭으로 포인트 찍어서 마스크 생성 (포함=da 내부, 제외=배경/**콘 위치**)
2. **ll(차선)**: YOLOPv2로 자동 추론 → 인스턴스 분리 + 스켈레톤화 → polyline 포인트로 변환
3. 위 두 결과를 합쳐서 **CVAT 1.1 XML**로 export → 로컬 CVAT에 import → 검수만 하면 끝
   (da만 먼저 팀원과 공유하고 싶으면 **COCO 1.0**으로 중간 export도 가능 — "중간 저장/이어하기" 다음 섹션 참고)

**LABELING_GUIDE.md에서 정한 규칙 (이 노트북이 지켜야 하는 것)**
- da는 좌/우 차선 사이 영역, 콘이 서 있는 자리는 da에서 제외
- 커브 등에서 차선이 한쪽만 보이면 반대쪽 경계는 화면 끝
- 역광/글레어는 흐릿해도 최대한 추정해서 그리되, 흔적이 전혀 없으면 비워둠
- ll은 polyline, 차선 하나당 점 3~6개, 곡선 구간은 촘촘히

⚠️ **주의**: 이 노트북은 `reasons.csv`에 있는 100장(cone/curve/failure) 기준으로 짜여 있음.
전체 데이터셋에 적용하려면 `image_files` 부분만 바꾸면 됨.

⚠️ **YOLOPv2는 BDD100K(실외 16:9 도로) 기준 학습 모델**이라 우리 트랙(실내, 다른 화면비 가능)에
그대로 돌리면 마스크가 원본 이미지에 정확히 안 맞을 수 있음 — 아래 "정렬 검증" 셀에서
반드시 눈으로 확인하고 넘어갈 것.

---
**현재 진행 상황** (마지막 업데이트 기준, 최신 상태는 `da_results.json` 확인)
- da(주행가능영역): **60 / 100장 완료** (SAM2 클릭 라벨링)
- ll(차선): **미착수** (YOLOPv2 pre-label 단계부터 시작하면 됨)
- 이어서 하려면: 설치 → Drive 마운트 → SAM2 로드까지 실행 후, "중간 저장/이어하기" 섹션의 `load_da_results()` 실행 → Gradio 앱 셀 재실행
---


In [ ]:
# ============================================================
# 0. 설치
# ============================================================
# SAM2 (facebookresearch/sam2) - pip 패키지가 아니라 레포를 클론해서 editable install
# ⚠️ 클론 폴더 이름을 'sam2'로 두면 안 됨: cwd(/content)에 'sam2' 폴더가 있으면
#    파이썬이 import sam2 할 때 이 폴더 자체를 패키지로 착각해서
#    "No module named 'sam2.build_sam'" 에러가 남 (SAM2 레포에 알려진 이슈).
#    그래서 폴더명을 sam2_repo로 지정해서 클론함.
!git clone -q https://github.com/facebookresearch/sam2.git sam2_repo
!pip -q install -e ./sam2_repo
# ⚠️ SAM2는 torch>=2.5.1 요구 - 설치 중 torch가 업그레이드되면 Colab 런타임을
#    재시작(런타임 > 세션 다시 시작)한 뒤 이 셀부터 다시 실행해야 할 수 있음

!pip -q install gradio opencv-python-headless scikit-image

# YOLOPv2 (CAIC-AD/YOLOPv2, 공식 저장소)
!git clone -q https://github.com/CAIC-AD/YOLOPv2.git
!mkdir -p YOLOPv2/data/weights
!wget -q https://github.com/CAIC-AD/YOLOPv2/releases/download/V0.0.1/yolopv2.pt \
    -O YOLOPv2/data/weights/yolopv2.pt

# SAM2.1 체크포인트 (hiera_small: tiny보다 정확하고 large보다 훨씬 가벼움 - Colab 무료 GPU용)
!mkdir -p sam2_repo/checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt \
    -O sam2_repo/checkpoints/sam2.1_hiera_small.pt

print("설치 완료")


In [ ]:
# ============================================================
# 1. Drive 마운트 + 경로 + reasons.csv 로드
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, csv

# ↓↓↓ 실제 Drive 경로로 수정 ↓↓↓
IMAGE_DIR = "/content/drive/MyDrive/kuac_lane_data/images"
REASONS_CSV = "/content/drive/MyDrive/kuac_lane_data/reasons.csv"
OUTPUT_DIR = "/content/drive/MyDrive/kuac_lane_data/cvat_export"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(REASONS_CSV) as f:
    reader = csv.DictReader(f)
    reasons_map = {row["file"]: row["reason"] for row in reader}

image_files = sorted(reasons_map.keys())
print(f"총 {len(image_files)}장 로드 (cone/curve/failure)")
assert len(image_files) == 100, "reasons.csv 100장과 안 맞음 - 경로 확인"

missing = [f for f in image_files if not os.path.exists(os.path.join(IMAGE_DIR, f))]
if missing:
    print(f"⚠️ IMAGE_DIR에 없는 파일 {len(missing)}개 (앞 5개): {missing[:5]}")
else:
    print("모든 이미지 파일 확인됨 (OK)")


## 유틸: 마스크 → polygon / polyline / CVAT XML

로컬에서 합성 데이터(사다리꼴 da, 곡선+직선 ll)로 이미 단위 테스트 통과한 코드 그대로임:
- `mask_to_polygon`: da 마스크(SAM 출력) → CVAT polygon 포인트
- `mask_to_polylines`: ll 마스크(YOLOPv2 출력) → 인스턴스 분리 + 스켈레톤 + 3~6점 샘플링
  (곡률 클수록 점 더 많이 — curve 카테고리에 특히 중요)
- `build_cvat_xml`: 위 둘을 합쳐 CVAT 1.1 XML(for images) 생성


In [ ]:
"""
da(drivable_area) 마스크 -> polygon 변환
ll(lane_line) 마스크 -> polyline 포인트 샘플링 (인스턴스 분리 + 스켈레톤화)
CVAT 1.1 XML(for images) 생성

Colab 노트북에서 SAM(da)/YOLOPv2(ll) 추론 결과를 이 모듈로 후처리해서
CVAT import용 XML로 합침.
"""
import cv2
import numpy as np
from skimage.morphology import skeletonize
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom


# ---------------------------------------------------------------------------
# 1. da: 이진 마스크 -> polygon 좌표 리스트
# ---------------------------------------------------------------------------
def mask_to_polygon(mask: np.ndarray, epsilon_ratio: float = 0.003):
    """
    SAM이 뱉은 da 바이너리 마스크(0/1 또는 0/255)를 CVAT polygon 포인트로 변환.
    - 가장 큰 컨투어만 사용 (SAM 마스크는 보통 단일 blob)
    - approxPolyDP로 점 개수를 줄여서 CVAT에서 다루기 편하게 함
    """
    m = (mask > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    largest = max(contours, key=cv2.contourArea)
    perimeter = cv2.arcLength(largest, True)
    epsilon = epsilon_ratio * perimeter
    approx = cv2.approxPolyDP(largest, epsilon, True)
    points = [(float(p[0][0]), float(p[0][1])) for p in approx]
    return points


# ---------------------------------------------------------------------------
# 2. ll: 이진 마스크(YOLOPv2 lane 출력) -> 인스턴스별 polyline 포인트 리스트
# ---------------------------------------------------------------------------
def mask_to_polylines(mask: np.ndarray, min_pts: int = 3, max_pts: int = 6,
                       min_component_area: int = 40):
    """
    YOLOPv2의 lane_line semantic mask(선들이 뭉쳐서 하나의 흰 영역)를
    개별 차선(polyline) 리스트로 분리.

    단계:
    1) connectedComponents로 서로 안 붙어있는 선 덩어리를 분리
    2) 각 덩어리를 skeletonize로 1px 중심선으로 축소
    3) 중심선을 따라 정렬한 뒤, 길이/곡률에 비례해 3~6개 점을 샘플링
       (직선 구간은 적게, 곡선 구간은 촘촘히 -> LABELING_GUIDE의 polyline 지침과 동일)

    returns: List[List[(x, y)]]  # 선 하나당 (x,y) 포인트 리스트
    """
    m = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)

    polylines = []
    for label_id in range(1, num_labels):  # 0 = background
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_component_area:
            continue  # 노이즈성 작은 덩어리 무시

        component_mask = (labels == label_id).astype(np.uint8)
        skeleton = skeletonize(component_mask.astype(bool))
        ys, xs = np.where(skeleton)
        if len(xs) < 2:
            continue

        ordered_pts = _order_skeleton_points(xs, ys)
        n_pts = _adaptive_point_count(ordered_pts, min_pts, max_pts)
        sampled = _sample_along_curve(ordered_pts, n_pts)
        polylines.append(sampled)

    return polylines


def _order_skeleton_points(xs, ys):
    """
    스켈레톤 픽셀은 순서가 없는 점 집합이라, 가장 가까운 이웃을 따라가며
    한쪽 끝에서 다른 쪽 끝까지 순서를 만들어줌 (그리디 nearest-neighbor).
    """
    pts = list(zip(xs.tolist(), ys.tolist()))
    if len(pts) <= 2:
        return pts

    pts_arr = np.array(pts, dtype=np.float64)
    # 가장 왼쪽(또는 위쪽) 점을 시작점으로
    start_idx = int(np.argmin(pts_arr[:, 0] + pts_arr[:, 1]))
    visited = np.zeros(len(pts_arr), dtype=bool)
    order = [start_idx]
    visited[start_idx] = True
    current = pts_arr[start_idx]

    for _ in range(len(pts_arr) - 1):
        remaining_idx = np.where(~visited)[0]
        if len(remaining_idx) == 0:
            break
        dists = np.linalg.norm(pts_arr[remaining_idx] - current, axis=1)
        nxt = remaining_idx[int(np.argmin(dists))]
        order.append(nxt)
        visited[nxt] = True
        current = pts_arr[nxt]

    return [tuple(pts_arr[i]) for i in order]


def _adaptive_point_count(ordered_pts, min_pts, max_pts):
    """곡률이 클수록(직선에서 많이 벗어날수록) 점을 더 많이 씀."""
    if len(ordered_pts) < 3:
        return min_pts
    pts = np.array(ordered_pts)
    start, end = pts[0], pts[-1]
    line_vec = end - start
    line_len = np.linalg.norm(line_vec)
    if line_len < 1e-6:
        return min_pts
    line_unit = line_vec / line_len
    # 각 점에서 시작-끝을 잇는 직선까지의 수직 거리 (직선성 척도)
    rel = pts - start
    proj_len = rel @ line_unit
    proj_point = np.outer(proj_len, line_unit) + start
    perp_dist = np.linalg.norm(pts - proj_point, axis=1)
    max_deviation = perp_dist.max()

    # 편차가 클수록(곡선일수록) max_pts에 가깝게, 직선이면 min_pts
    curviness = np.clip(max_deviation / max(line_len * 0.05, 1.0), 0.0, 1.0)
    n_pts = int(round(min_pts + curviness * (max_pts - min_pts)))
    return max(min_pts, min(max_pts, n_pts))


def _sample_along_curve(ordered_pts, n_pts):
    """정렬된 스켈레톤 점들에서 누적 호 길이 기준 등간격으로 n_pts개 샘플링."""
    pts = np.array(ordered_pts, dtype=np.float64)
    if len(pts) <= n_pts:
        return [tuple(p) for p in pts]

    seg_lens = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    cum_len = np.concatenate([[0], np.cumsum(seg_lens)])
    total_len = cum_len[-1]
    if total_len < 1e-6:
        return [tuple(pts[0])] * n_pts

    targets = np.linspace(0, total_len, n_pts)
    sampled = []
    for t in targets:
        idx = int(np.searchsorted(cum_len, t))
        idx = min(max(idx, 1), len(cum_len) - 1)
        t0, t1 = cum_len[idx - 1], cum_len[idx]
        p0, p1 = pts[idx - 1], pts[idx]
        ratio = 0.0 if t1 == t0 else (t - t0) / (t1 - t0)
        sampled.append(tuple(p0 + ratio * (p1 - p0)))
    return sampled


# ---------------------------------------------------------------------------
# 3. CVAT 1.1 XML 생성
# ---------------------------------------------------------------------------
def build_cvat_xml(images_annotations, pretty=True):
    """
    images_annotations: List[dict], 각 dict는:
        {
            "id": int,
            "name": str,           # 파일명, 예: frame_000125.png
            "width": int,
            "height": int,
            "da_polygon": [(x,y), ...] or None,
            "ll_polylines": [[(x,y),...], [(x,y),...], ...],
        }
    returns: XML 문자열 (CVAT "Import annotations -> CVAT 1.1" 포맷)
    """
    root = Element("annotations")
    SubElement(root, "version").text = "1.1"

    for img in images_annotations:
        image_el = SubElement(root, "image", {
            "id": str(img["id"]),
            "name": img["name"],
            "width": str(img["width"]),
            "height": str(img["height"]),
        })

        if img.get("da_polygon"):
            pts_str = ";".join(f"{x:.2f},{y:.2f}" for x, y in img["da_polygon"])
            SubElement(image_el, "polygon", {
                "label": "drivable_area",
                "points": pts_str,
                "occluded": "0",
            })

        for line_pts in img.get("ll_polylines", []):
            pts_str = ";".join(f"{x:.2f},{y:.2f}" for x, y in line_pts)
            SubElement(image_el, "polyline", {
                "label": "lane_line",
                "points": pts_str,
                "occluded": "0",
            })

    xml_bytes = tostring(root, encoding="utf-8")
    if pretty:
        return minidom.parseString(xml_bytes).toprettyxml(indent="  ")
    return xml_bytes.decode("utf-8")


## 1단계 — da(주행가능영역): SAM 클릭 라벨링

- **좌클릭 모드 "포함 (da 내부)"**: da 영역 안쪽에 몇 번 클릭
- **모드를 "제외 (배경/콘)"로 바꾸고 클릭**: 차선 밖, 그리고 **콘이 서 있는 자리**를 클릭
  → SAM이 그 지점을 마스크에서 자동으로 빼줌 (가이드 3장 "콘 자리는 da 제외" 규칙과 동일)
- 커브라서 차선이 한쪽만 보이면, 안 보이는 쪽은 화면 끝까지 포함하도록 포인트를 찍으면 됨
- "마스크 생성"으로 미리보기 → 맘에 들면 "저장하고 다음" → 다음 이미지로 자동 이동
- 결과는 `da_results` 딕셔너리(파일명 → polygon 점 리스트)에 누적됨


In [ ]:
# ============================================================
# 2. SAM2 모델 로드
# ============================================================
import numpy as np
import cv2
import torch
from contextlib import nullcontext

# editable install(pip -e)로 방금 깐 패키지가 같은 세션의 커널에 바로 안 잡힐 때가 있어서
# sam2_repo 경로를 sys.path에 직접 넣어줌 (안전장치, 이미 잡혀있으면 무해함)
import sys
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

sam2_checkpoint = "sam2_repo/checkpoints/sam2.1_hiera_small.pt"
sam2_model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(sam2_model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

def sam_autocast():
    # GPU에서만 autocast 사용 (CPU면 그냥 기본 정밀도로)
    if device == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

print("SAM2 (hiera_small) 로드 완료")


In [ ]:
# ============================================================
# 3. da 클릭 라벨링 Gradio 앱
# ============================================================
import gradio as gr

state = {"idx": 0, "points": [], "current_mask": None}
da_results = {}  # 이미 진행한 게 있으면 아래 '이어하기' 셀로 불러와서 여기에 채우기

def _load_current_image():
    fname = image_files[state["idx"]]
    img_bgr = cv2.imread(os.path.join(IMAGE_DIR, fname))
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def _draw_overlay(img_rgb, points, mask=None):
    vis = img_rgb.copy()
    if mask is not None:
        overlay = vis.copy()
        overlay[mask > 0] = (0, 255, 0)
        vis = cv2.addWeighted(overlay, 0.4, vis, 0.6, 0)
    for x, y, lbl in points:
        color = (0, 200, 255) if lbl == 1 else (255, 0, 0)
        cv2.circle(vis, (int(x), int(y)), 6, color, -1)
    return vis

def _status_text():
    fname = image_files[state['idx']]
    return f"{state['idx']+1} / {len(image_files)} - {fname} ({reasons_map.get(fname, '?')}) | 완료: {len(da_results)}"

def on_select(point_mode, evt: gr.SelectData):
    x, y = evt.index
    label = 1 if point_mode == "포함 (da 내부)" else 0
    state["points"].append((x, y, label))
    return _draw_overlay(_load_current_image(), state["points"], state["current_mask"])

def run_sam():
    if not state["points"]:
        return _load_current_image()
    img_rgb = _load_current_image()
    coords = np.array([[p[0], p[1]] for p in state["points"]])
    labels = np.array([p[2] for p in state["points"]])
    with torch.inference_mode(), sam_autocast():
        predictor.set_image(img_rgb)
        masks, scores, _ = predictor.predict(point_coords=coords, point_labels=labels, multimask_output=True)
    best = masks[int(np.argmax(scores))]
    state["current_mask"] = best
    return _draw_overlay(img_rgb, state["points"], best)

def reset_points():
    state["points"] = []
    state["current_mask"] = None
    return _load_current_image()

def accept_and_next():
    fname = image_files[state["idx"]]
    if state["current_mask"] is not None:
        polygon = mask_to_polygon(state["current_mask"].astype(np.uint8))
        da_results[fname] = polygon
    state["idx"] = min(state["idx"] + 1, len(image_files) - 1)
    state["points"], state["current_mask"] = [], None
    return _load_current_image(), _status_text()

def skip_next():
    state["idx"] = min(state["idx"] + 1, len(image_files) - 1)
    state["points"], state["current_mask"] = [], None
    return _load_current_image(), _status_text()

def prev_image():
    state["idx"] = max(state["idx"] - 1, 0)
    state["points"], state["current_mask"] = [], None
    return _load_current_image(), _status_text()

with gr.Blocks() as demo:
    gr.Markdown("### da 클릭 라벨링 — 포함(전경)/제외(배경,콘) 클릭 후 '마스크 생성'")
    point_mode = gr.Radio(["포함 (da 내부)", "제외 (배경/콘)"],
                           value="포함 (da 내부)", label="클릭 모드")
    img_display = gr.Image(value=_load_current_image(), interactive=True)
    status = gr.Textbox(value=_status_text(), label="진행상황", interactive=False)
    with gr.Row():
        gen_btn = gr.Button("마스크 생성")
        reset_btn = gr.Button("포인트 초기화")
        prev_btn = gr.Button("← 이전")
        skip_btn = gr.Button("건너뛰기 →")
        accept_btn = gr.Button("저장하고 다음 →", variant="primary")

    img_display.select(on_select, inputs=[point_mode], outputs=[img_display])
    gen_btn.click(run_sam, outputs=[img_display])
    reset_btn.click(reset_points, outputs=[img_display])
    accept_btn.click(accept_and_next, outputs=[img_display, status])
    skip_btn.click(skip_next, outputs=[img_display, status])
    prev_btn.click(prev_image, outputs=[img_display, status])

demo.launch(share=True, debug=False)


## 중간 저장 / 이어하기

Colab 세션이 끊길 수 있으니, 중간중간 저장해두고 다음에 이어서 하면 됨.

In [ ]:
# ============================================================
# 4. da_results 저장 / 불러오기
# ============================================================
import json

DA_RESULTS_PATH = os.path.join(OUTPUT_DIR, "da_results.json")

def save_da_results():
    serializable = {k: [[float(x), float(y)] for x, y in v] for k, v in da_results.items()}
    with open(DA_RESULTS_PATH, "w") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)
    print(f"{len(serializable)}장 저장 -> {DA_RESULTS_PATH}")

def load_da_results():
    if not os.path.exists(DA_RESULTS_PATH):
        print("저장된 파일 없음 (처음 시작)")
        return
    with open(DA_RESULTS_PATH) as f:
        loaded = json.load(f)
    for k, v in loaded.items():
        da_results[k] = [(x, y) for x, y in v]
    # 이어서 할 때는 완료된 것 다음 인덱스부터 시작
    done = set(da_results.keys())
    for i, f in enumerate(image_files):
        if f not in done:
            state["idx"] = i
            break
    print(f"{len(loaded)}장 불러옴, {state['idx']+1}번째 이미지부터 이어서 진행")

# 저장: save_da_results()
# 이어서 시작하려면 위 Gradio 셀 재실행 전에: load_da_results()


In [ ]:
save_da_results()

### da 결과 내보내기 — COCO 1.0 (da 전용, 팀원 공유용)

여기서 만든 `da_coco.json`은 **현재까지 완료된 da만** 담김 — ll은 다음 "2단계"에서 만드는데, COCO 포맷은 polyline을 지원하지 않아서 ll은 이 파일에 못 넣음. 지금 상태를 빨리 팀원과 공유하고 싶을 때 쓰는 중간 산출물이고, **최종본은 da+ll을 합친 CVAT XML**(3단계)이 될 예정.

CVAT에서 열 때: Task 생성 → 이미지 업로드 → Job에서 `Actions → Upload annotations` → 포맷 **"COCO 1.0"** 선택 → `da_coco.json` 업로드.

In [ ]:
# ============================================================
# da_results -> COCO 1.0 export (팀원 공유용, da만 담김 - ll은 polyline이라 COCO 불가)
# ============================================================
import json

def _polygon_area(points):
    n = len(points)
    area = 0.0
    for i in range(n):
        x1, y1 = points[i]
        x2, y2 = points[(i + 1) % n]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2.0

def build_coco_annotations(images_annotations, category_name="drivable_area"):
    coco = {"images": [], "annotations": [],
            "categories": [{"id": 1, "name": category_name, "supercategory": ""}]}
    ann_id = 1
    for img in images_annotations:
        coco["images"].append({"id": img["id"], "file_name": img["name"],
                                "width": img["width"], "height": img["height"]})
        polygon = img.get("da_polygon")
        if not polygon or len(polygon) < 3:
            continue
        xs = [p[0] for p in polygon]; ys = [p[1] for p in polygon]
        bbox = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
        seg = [c for p in polygon for c in p]
        coco["annotations"].append({"id": ann_id, "image_id": img["id"], "category_id": 1,
                                     "segmentation": [seg], "area": _polygon_area(polygon),
                                     "bbox": bbox, "iscrowd": 0})
        ann_id += 1
    return coco

images_annotations = []
for i, fname in enumerate(image_files):
    img_bgr = cv2.imread(os.path.join(IMAGE_DIR, fname))
    h, w = img_bgr.shape[:2]
    images_annotations.append({
        "id": i, "name": fname, "width": w, "height": h,
        "da_polygon": da_results.get(fname),
    })

coco = build_coco_annotations(images_annotations)
COCO_PATH = os.path.join(OUTPUT_DIR, "da_coco.json")
with open(COCO_PATH, "w") as f:
    json.dump(coco, f)

n_done = len(coco["annotations"])
print(f"da 완료 {n_done}장 -> COCO annotation 생성, 전체 이미지 {len(coco['images'])}장 포함")
print(f"저장 -> {COCO_PATH}")

### COCO 파일 정합성 검증 (선택, 공유 전 확인용)

`da_coco.json`에 기록된 이미지 크기가 실제 파일과 일치하는지, polygon 좌표가 이미지 범위를 벗어나지 않는지 확인. 문제 없으면 그대로 공유해도 됨.

In [ ]:
def check_coco_consistency(coco_path, image_dir):
    with open(coco_path) as f:
        coco = json.load(f)
    img_dims = {img["id"]: (img["width"], img["height"], img["file_name"]) for img in coco["images"]}
    problems = []
    for img in coco["images"]:
        real_path = os.path.join(image_dir, img["file_name"])
        if not os.path.exists(real_path):
            problems.append(f"{img['file_name']}: 이미지 파일 없음")
            continue
        real = cv2.imread(real_path)
        rh, rw = real.shape[:2]
        if (rw, rh) != (img["width"], img["height"]):
            problems.append(f"{img['file_name']}: JSON 기록 크기 {img['width']}x{img['height']} != 실제 파일 {rw}x{rh}")
    for ann in coco["annotations"]:
        w, h, fname = img_dims[ann["image_id"]]
        seg = ann["segmentation"][0]
        xs, ys = seg[0::2], seg[1::2]
        if min(xs) < 0 or max(xs) > w or min(ys) < 0 or max(ys) > h:
            problems.append(f"{fname}: polygon 좌표가 이미지 범위 벗어남 (x:{min(xs):.1f}~{max(xs):.1f}/w={w}, y:{min(ys):.1f}~{max(ys):.1f}/h={h})")
    return problems

problems = check_coco_consistency(COCO_PATH, IMAGE_DIR)
if not problems:
    print(f"✅ 문제 없음 — 이미지 크기 100장 전부 일치, polygon 좌표도 다 범위 안")
else:
    print(f"⚠️ {len(problems)}건 발견:")
    for p in problems:
        print(" -", p)

## 2단계 — ll(차선): YOLOPv2 자동 pre-label

YOLOPv2가 뱉는 lane_line 마스크(선들이 뭉친 흰 영역)를 그대로 쓰지 않고,
`mask_to_polylines`로 인스턴스 분리 + 스켈레톤화 + 점 샘플링까지 자동 처리함.

**먼저 2~3장으로 마스크가 원본 이미지에 제대로 정렬되는지 눈으로 확인하고 나서**
전체 100장 배치로 넘어갈 것 — YOLOPv2는 BDD100K(16:9) 기준이라 화면비가 다르면
어긋날 수 있음.


In [ ]:
# ============================================================
# 5. YOLOPv2 모델 로드
# ============================================================
import sys
sys.path.insert(0, "YOLOPv2")
from utils.utils import driving_area_mask, lane_line_mask, LoadImages

yolop_device = torch.device(device)
yolop_half = yolop_device.type != "cpu"

yolopv2_model = torch.jit.load("YOLOPv2/data/weights/yolopv2.pt")
yolopv2_model = yolopv2_model.to(yolop_device)
if yolop_half:
    yolopv2_model.half()
yolopv2_model.eval()
print("YOLOPv2 로드 완료")


In [ ]:
# ============================================================
# 6. 정렬 검증 (필수 — 샘플 3장)
# ============================================================
import matplotlib.pyplot as plt

def run_yolopv2_on_image(img_path):
    dataset = LoadImages(img_path, img_size=640, stride=32)
    path, img, im0s, _ = next(iter(dataset))
    img_t = torch.from_numpy(img).to(yolop_device)
    img_t = img_t.half() if yolop_half else img_t.float()
    img_t /= 255.0
    if img_t.ndimension() == 3:
        img_t = img_t.unsqueeze(0)
    with torch.no_grad():
        [pred, anchor_grid], seg, ll = yolopv2_model(img_t)
    da_mask = driving_area_mask(seg)
    ll_mask = lane_line_mask(ll)
    return im0s, da_mask, ll_mask

sample_files = image_files[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, fname in zip(axes, sample_files):
    im0s, da_mask, ll_mask = run_yolopv2_on_image(os.path.join(IMAGE_DIR, fname))
    vis = cv2.cvtColor(im0s, cv2.COLOR_BGR2RGB).copy()
    print(fname, "원본:", im0s.shape[:2], "| da_mask:", da_mask.shape, "| ll_mask:", ll_mask.shape)
    if ll_mask.shape[:2] == im0s.shape[:2]:
        vis[ll_mask > 0] = (255, 0, 0)
    else:
        print("  ⚠️ ll_mask 크기가 원본과 다름 - 아래 resize 필요")
        ll_resized = cv2.resize(ll_mask.astype(np.uint8), (im0s.shape[1], im0s.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)
        vis[ll_resized > 0] = (255, 0, 0)
    ax.imshow(vis)
    ax.set_title(fname)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("\n>>> 빨간 오버레이가 실제 차선 위에 잘 얹히는지 눈으로 확인! "
      "어긋나 보이면 다음 셀로 넘어가지 말고 알려줘.")


In [ ]:
# ============================================================
# 7. 전체 100장 배치 추론 + polyline 변환
# ============================================================
ll_results = {}  # 파일명 -> [[(x,y),...], [(x,y),...], ...]

for fname in image_files:
    im0s, da_mask, ll_mask = run_yolopv2_on_image(os.path.join(IMAGE_DIR, fname))
    if ll_mask.shape[:2] != im0s.shape[:2]:
        ll_mask = cv2.resize(ll_mask.astype(np.uint8), (im0s.shape[1], im0s.shape[0]),
                              interpolation=cv2.INTER_NEAREST)
    polylines = mask_to_polylines(ll_mask, min_pts=3, max_pts=6)
    ll_results[fname] = polylines
    print(f"{fname} ({reasons_map.get(fname)}): 차선 {len(polylines)}개")

print(f"\n총 {len(ll_results)}장 처리 완료")

# 저장
LL_RESULTS_PATH = os.path.join(OUTPUT_DIR, "ll_results.json")
serializable = {
    k: [[[float(x), float(y)] for x, y in line] for line in v]
    for k, v in ll_results.items()
}
with open(LL_RESULTS_PATH, "w") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)
print(f"저장 -> {LL_RESULTS_PATH}")


## 3단계 — da + ll 합쳐서 CVAT XML로 export

주의: ll은 YOLOPv2 pre-label(초안)이라 특히 `curve`/`failure` 카테고리는
CVAT에서 열었을 때 사람이 많이 고쳐야 할 수 있음 — 가이드 4장에서 이미 말한 부분.
da는 안 끝낸 이미지가 있으면 그 이미지는 da 없이 export됨 (CVAT에서 마저 그리면 됨).


In [ ]:
# ============================================================
# 8. CVAT XML 생성
# ============================================================
if not da_results:
    load_da_results()

images_annotations = []
for i, fname in enumerate(image_files):
    img_bgr = cv2.imread(os.path.join(IMAGE_DIR, fname))
    h, w = img_bgr.shape[:2]
    images_annotations.append({
        "id": i,
        "name": fname,
        "width": w,
        "height": h,
        "da_polygon": da_results.get(fname),
        "ll_polylines": ll_results.get(fname, []),
    })

xml_str = build_cvat_xml(images_annotations)

XML_PATH = os.path.join(OUTPUT_DIR, "cvat_annotations.xml")
with open(XML_PATH, "w", encoding="utf-8") as f:
    f.write(xml_str)

n_da = sum(1 for img in images_annotations if img["da_polygon"])
n_ll = sum(1 for img in images_annotations if img["ll_polylines"])
print(f"da 완료: {n_da}/{len(image_files)}, ll 완료: {n_ll}/{len(image_files)}")
print(f"XML 저장 -> {XML_PATH}")


In [ ]:
# ============================================================
# 9. CVAT import용 zip 묶기 (이미지 + annotations.xml)
# ============================================================
import shutil, zipfile

EXPORT_ZIP_DIR = os.path.join(OUTPUT_DIR, "cvat_import")
os.makedirs(EXPORT_ZIP_DIR, exist_ok=True)

for fname in image_files:
    shutil.copy(os.path.join(IMAGE_DIR, fname), os.path.join(EXPORT_ZIP_DIR, fname))
shutil.copy(XML_PATH, os.path.join(EXPORT_ZIP_DIR, "annotations.xml"))

ZIP_PATH = os.path.join(OUTPUT_DIR, "cvat_import.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(EXPORT_ZIP_DIR):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.relpath(fp, EXPORT_ZIP_DIR))

print(f"완료 -> {ZIP_PATH}")
print("CVAT에서: Task 생성 시 이미지 100장 업로드 -> Job 열고 "
      "'Actions > Upload annotations' -> 포맷 'CVAT 1.1' 선택 -> annotations.xml 업로드")
